# 🚀 Stage 4: CUDA & GPU Performance Engineering Masterclass
### Interactive Hands-on Notebook for Google Colab (Tesla T4) & RTX 3090

This notebook contains all live milestones for our low-level CUDA learning:
- **Milestone 0:** GPU Silicon & CPU vs GPU Throughput Crossover
- **Milestone 4.1:** Your First CUDA Kernel (Naive Matrix Multiplication)
- **Milestone 4.2:** Shared Memory Tiled Matrix Multiplication (SRAM Caching)
- **Lesson 3:** Memory Coalescing Penalties & Shared Memory Bank Conflicts

## 📦 Step 0: Install Dependencies & Verify GPU

In [ ]:
!pip install -q ninja
!nvidia-smi

--- 
## 🧠 Milestone 0: GPU Properties & CPU vs GPU Throughput
Let's query the physical SM count, VRAM, and compare when CPU wins (small data) vs when GPU wins (massive parallel data).

In [ ]:
import torch
import time

device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)

print("=" * 70)
print("GPU HARDWARE SPECS")
print("=" * 70)
print(f"GPU Name:                {props.name}")
print(f"Compute Capability:      {props.major}.{props.minor}")
print(f"Total VRAM (Global Mem): {props.total_memory / 1e9:.1f} GB")
print(f"Streaming Multiprocessors (SMs): {props.multi_processor_count}")
print(f"CUDA Cores (Est):        {props.multi_processor_count * 64}")
print(f"Warp Size:               32 threads (NVIDIA constant)")
print("=" * 70)

# Benchmark Vector Addition CPU vs GPU
sizes = [100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000]
print(f"\n{'Vector Size':>12} | {'CPU (ms)':>10} | {'GPU (ms)':>10} | {'Speedup':>10} | {'Winner':>8}")
print("-" * 60)

for N in sizes:
    a_cpu = torch.randn(N, dtype=torch.float32)
    b_cpu = torch.randn(N, dtype=torch.float32)
    
    # CPU Timing
    t0 = time.perf_counter()
    c_cpu = a_cpu + b_cpu
    t_cpu = (time.perf_counter() - t0) * 1000
    
    # GPU Timing
    a_gpu = a_cpu.cuda()
    b_gpu = b_cpu.cuda()
    torch.cuda.synchronize()
    
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    start.record()
    c_gpu = a_gpu + b_gpu
    end.record()
    torch.cuda.synchronize()
    t_gpu = start.elapsed_time(end)
    
    speedup = t_cpu / t_gpu if t_gpu > 0 else 0
    winner = "GPU 🚀" if speedup > 1.0 else "CPU 💻"
    print(f"{N:>12,d} | {t_cpu:>10.3f} | {t_gpu:>10.3f} | {speedup:>9.1f}x | {winner:>8}")

--- 
## ⚡ Milestone 4.1: Your First Raw CUDA C++ Kernel (Naive Matmul)
Here we write a real raw CUDA C++ kernel using `__global__` and compile it with `nvcc` via PyTorch's `load_inline`.

In [ ]:
import torch
from torch.utils.cpp_extension import load_inline

cuda_naive = '''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

__global__ void naive_matmul_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0f; // Register
        for (int k = 0; k < K; ++k) {
            sum += A[row * K + k] * B[k * N + col]; // Global VRAM read every step!
        }
        C[row * N + col] = sum;
    }
}

void run_naive(torch::Tensor A, torch::Tensor B, torch::Tensor C) {
    int M = A.size(0);
    int K = A.size(1);
    int N = B.size(1);

    dim3 block(16, 16); // 256 threads
    dim3 grid((N + 15) / 16, (M + 15) / 16);

    naive_matmul_kernel<<<grid, block>>>(
        A.data_ptr<float>(),
        B.data_ptr<float>(),
        C.data_ptr<float>(),
        M, N, K
    );
}
'''

cpp_naive = 'void run_naive(torch::Tensor A, torch::Tensor B, torch::Tensor C);'

naive_module = load_inline(
    name='naive_matmul',
    cpp_sources=cpp_naive,
    cuda_sources=cuda_naive,
    functions=['run_naive'],
    extra_cuda_cflags=['-O3']
)
print("✅ Naive CUDA Kernel Compiled Successfully!")

In [ ]:
# Benchmark Naive vs cuBLAS
def benchmark(fn, *args, iters=10):
    for _ in range(3): fn(*args)
    torch.cuda.synchronize()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iters): fn(*args)
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / iters

sizes = [128, 256, 512, 1024, 2048]
print(f"{'Matrix Size':>12} | {'cuBLAS (ms)':>12} | {'Naive (ms)':>12} | {'cuBLAS vs Naive':>18} | {'Correct?':>8}")
print("-" * 72)

for N in sizes:
    A = torch.randn(N, N, device='cuda')
    B = torch.randn(N, N, device='cuda')
    C = torch.empty(N, N, device='cuda')
    
    t_cublas = benchmark(torch.mm, A, B)
    t_naive = benchmark(naive_module.run_naive, A, B, C)
    
    is_correct = torch.allclose(C, A @ B, atol=1e-3, rtol=1e-3)
    correct_str = "✅ YES" if is_correct else "❌ NO"
    speedup = t_naive / t_cublas
    print(f"{f'{N}x{N}':>12} | {t_cublas:>12.3f} | {t_naive:>12.3f} | {speedup:>17.1f}x slower | {correct_str:>8}")

--- 
## 🏢 Milestone 4.2: Shared Memory Tiled Matrix Multiplication (L1 SRAM Caching)
Now we use `__shared__ float s_A[16][16]` and `__syncthreads()` to cache chunks into on-chip SRAM.

In [ ]:
cuda_tiled = '''
#include <torch/extension.h>
#include <cuda.h>
#include <cuda_runtime.h>

#define TILE_SIZE 16

__global__ void tiled_matmul_kernel(
    const float* __restrict__ A,
    const float* __restrict__ B,
    float* __restrict__ C,
    int M, int N, int K)
{
    __shared__ float s_A[TILE_SIZE][TILE_SIZE];
    __shared__ float s_B[TILE_SIZE][TILE_SIZE];

    int tx = threadIdx.x;
    int ty = threadIdx.y;
    int row = blockIdx.y * TILE_SIZE + ty;
    int col = blockIdx.x * TILE_SIZE + tx;

    float acc = 0.0f;
    int num_tiles = (K + TILE_SIZE - 1) / TILE_SIZE;

    for (int t = 0; t < num_tiles; ++t) {
        int a_col = t * TILE_SIZE + tx;
        int b_row = t * TILE_SIZE + ty;

        s_A[ty][tx] = (row < M && a_col < K) ? A[row * K + a_col] : 0.0f;
        s_B[ty][tx] = (b_row < K && col < N) ? B[b_row * N + col] : 0.0f;

        __syncthreads(); // Wait for tile load

        #pragma unroll
        for (int k = 0; k < TILE_SIZE; ++k) {
            acc += s_A[ty][k] * s_B[k][tx];
        }

        __syncthreads(); // Wait before next tile load
    }

    if (row < M && col < N) {
        C[row * N + col] = acc;
    }
}

void run_tiled(torch::Tensor A, torch::Tensor B, torch::Tensor C) {
    int M = A.size(0);
    int K = A.size(1);
    int N = B.size(1);

    dim3 block(TILE_SIZE, TILE_SIZE);
    dim3 grid((N + TILE_SIZE - 1) / TILE_SIZE, (M + TILE_SIZE - 1) / TILE_SIZE);

    tiled_matmul_kernel<<<grid, block>>>(
        A.data_ptr<float>(),
        B.data_ptr<float>(),
        C.data_ptr<float>(),
        M, N, K
    );
}
'''

cpp_tiled = 'void run_tiled(torch::Tensor A, torch::Tensor B, torch::Tensor C);'

tiled_module = load_inline(
    name='tiled_matmul',
    cpp_sources=cpp_tiled,
    cuda_sources=cuda_tiled,
    functions=['run_tiled'],
    extra_cuda_cflags=['-O3']
)
print("✅ Shared Memory Tiled CUDA Kernel Compiled Successfully!")

In [ ]:
# Side-by-side comparison: cuBLAS vs Naive vs Tiled
print(f"{'Matrix Size':>12} | {'cuBLAS (ms)':>12} | {'Naive (ms)':>12} | {'Tiled (ms)':>12} | {'Tiled Speedup':>15}")
print("-" * 75)

for N in sizes:
    A = torch.randn(N, N, device='cuda')
    B = torch.randn(N, N, device='cuda')
    C_naive = torch.empty(N, N, device='cuda')
    C_tiled = torch.empty(N, N, device='cuda')
    
    t_cublas = benchmark(torch.mm, A, B)
    t_naive = benchmark(naive_module.run_naive, A, B, C_naive)
    t_tiled = benchmark(tiled_module.run_tiled, A, B, C_tiled)
    
    speedup = t_naive / t_tiled
    print(f"{f'{N}x{N}':>12} | {t_cublas:>12.3f} | {t_naive:>12.3f} | {t_tiled:>12.3f} | {speedup:>14.2f}x 🚀")

--- 
## 🔬 Lesson 3: Live Memory Coalescing & Bank Conflict Experiments

In [ ]:
# Run the dedicated Lesson 3 memory benchmark script
!python3 ai_ms_python/learn_projects/cuda_kernels/03_coalescing_and_bank_conflicts.py